In [2]:
# ============================================
# Module 2 : Data Cleaning and Structuring
# Final cleaned dataset for analysis & Tableau
# ============================================

# Import required libraries
import pandas as pd
import re

# --------------------------------------------
# Step 1: Load the raw dataset
# --------------------------------------------
df = pd.read_csv("global_military_data_final.csv")

# --------------------------------------------
# Step 2: Standardize column names
# Rules:
# - lowercase
# - replace spaces with underscores
# - remove special characters
# --------------------------------------------
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("%", "percent")
    .str.replace(r"[^a-z0-9_]", "", regex=True)
)

# --------------------------------------------
# Step 3: Remove capital-level population column
# This column is not required for country-level
# military analytics
# --------------------------------------------
df.drop(
    columns=["capitalcitiesbytotalpopulation"],
    errors="ignore",
    inplace=True
)

# --------------------------------------------
# Step 4: Identify categorical (string) columns
# Alphanumeric columns are treated as STRINGS
# --------------------------------------------
categorical_cols = ["country", "region"]

# --------------------------------------------
# Step 5: Clean numeric text values
# Removes commas, %, +, ~ and other symbols
# --------------------------------------------
def clean_numeric(value):
    if pd.isna(value):
        return None
    value = str(value)
    value = re.sub(r"[,%+~]", "", value)
    return value.strip()

# Apply cleaning to all non-categorical columns
for col in df.columns:
    if col not in categorical_cols:
        df[col] = df[col].apply(clean_numeric)

# --------------------------------------------
# Step 6: Convert cleaned columns to numeric
# --------------------------------------------
for col in df.columns:
    if col not in categorical_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# --------------------------------------------
# Step 7: Handle missing / null values
# Algorithms used:
# - Zero Imputation (for count-based metrics)
# - Median Imputation (for other numeric data)
# --------------------------------------------
count_keywords = ["tank", "aircraft", "naval", "vehicle", "artillery"]

for col in df.columns:
    if col not in categorical_cols:
        if any(keyword in col for keyword in count_keywords):
            df[col] = df[col].fillna(0)              # Zero Imputation
        else:
            df[col] = df[col].fillna(df[col].median())  # Median Imputation

# --------------------------------------------
# Step 8: Verify missing value percentage
# --------------------------------------------
missing_percentage = (df.isna().sum() / len(df)) * 100
print("Missing value percentage per column:")
print(missing_percentage)

# --------------------------------------------
# Step 9: Save FINAL cleaned dataset
# --------------------------------------------
df.to_csv("military_cleaned_final.csv", index=False)

print("✅ Data cleaning completed successfully")
print("📁 Output file saved as: military_cleaned_final.csv")


Missing value percentage per column:
country                                0.0
rank                                   0.0
totalpopulationbycountry               0.0
availablemilitarymanpower              0.0
manpowerfitformilitaryservice          0.0
manpowerreachingmilitaryageannually    0.0
activemilitarymanpower                 0.0
activereservemilitarymanpower          0.0
manpowerparamilitary                   0.0
aircrafttotal                          0.0
aircrafttotalfighters                  0.0
aircrafttotalattacktypes               0.0
aircrafttotaltransports                0.0
aircrafttotaltrainers                  0.0
aircrafttotalspecialmission            0.0
aircrafttotaltankerfleet               0.0
aircrafthelicopterstotal               0.0
aircrafthelicoptersattack              0.0
armortankstotal                        0.0
armorapctotal                          0.0
armorselfpropelledgunstotal            0.0
armortowedartillerytotal               0.0
armormlrstotal   